# L-Neurons — Secret Loyalties Hackathon

White-box sparse probes for secret-loyalty detection (Apart × Formation, Jul 2026).

This notebook:
1. Clones / uses the `J-nuerons` repo
2. Runs the deterministic **mock** pipeline (CPU)
3. Optionally runs the live **Qwen2.5-1.5B-Instruct** pipeline (GPU)

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO = Path('/content/J-nuerons')
if not REPO.exists():
    # Local / cloud-agent fallback
    local = Path.cwd()
    if (local / 'secret-loyalties').exists():
        REPO = local
    elif (local.parent / 'secret-loyalties').exists():
        REPO = local.parent
    else:
        subprocess.check_call(['git', 'clone', 'https://github.com/TinevimboMusingadi/J-nuerons.git', str(REPO)])

os.chdir(REPO)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-learn', 'numpy', 'tqdm'])
print('Repo ready:', REPO)

In [ ]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / 'secret-loyalties' / 'src'))

from run_pipeline import run_mock
from dataset_builder import load_scenarios

doc = load_scenarios(REPO / 'secret-loyalties' / 'data' / 'scenarios.json')
report = run_mock(doc, REPO / 'secret-loyalties' / 'results')
print(json.dumps({
    'in_domain': {k: {'auroc': v['auroc'], 'n_l_neurons': v['n_l_neurons']} for k, v in report['whitebox_in_domain'].items()},
    'transfer': report['cross_principal_transfer'],
    'jaccard': report['neuron_jaccard'],
}, indent=2))

In [ ]:
RUN_LIVE = False  # set True on a GPU runtime

if RUN_LIVE:
    from run_pipeline import run_model
    live = run_model(doc, REPO / 'secret-loyalties' / 'results', 'Qwen/Qwen2.5-1.5B-Instruct')
    print(json.dumps(live['whitebox_in_domain'], indent=2)[:2000])
else:
    print('Skipping live model run. Set RUN_LIVE = True to execute.')